In [1]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt


In [2]:
dataset_path = os.listdir("/content/drive/MyDrive/Dataset/rooms_dataset")

In [3]:
room_type = os.listdir("/content/drive/MyDrive/Dataset/rooms_dataset")
print(room_type)
print("types of room found: ", len(room_type))

['living_room', 'dining_room', 'bed_room']
types of room found:  3


In [4]:
rooms = []

for item in room_type:
    #get all rooms

    all_rooms = os.listdir("/content/drive/MyDrive/Dataset/rooms_dataset" +"/"+ item)

    # adding them to list

    for img in all_rooms:
        rooms.append((item, str("/content/drive/MyDrive/Dataset/rooms_dataset"+"/"+item) +"/"+img))
        print(rooms)

[('living_room', '/content/drive/MyDrive/Dataset/rooms_dataset/living_room/pexels-photo-1571453.jpeg')]
[('living_room', '/content/drive/MyDrive/Dataset/rooms_dataset/living_room/pexels-photo-1571453.jpeg'), ('living_room', '/content/drive/MyDrive/Dataset/rooms_dataset/living_room/photo-1554995207-c18c203602cb (1).jpg')]
[('living_room', '/content/drive/MyDrive/Dataset/rooms_dataset/living_room/pexels-photo-1571453.jpeg'), ('living_room', '/content/drive/MyDrive/Dataset/rooms_dataset/living_room/photo-1554995207-c18c203602cb (1).jpg'), ('living_room', '/content/drive/MyDrive/Dataset/rooms_dataset/living_room/pexels-photo-1571470.jpeg')]
[('living_room', '/content/drive/MyDrive/Dataset/rooms_dataset/living_room/pexels-photo-1571453.jpeg'), ('living_room', '/content/drive/MyDrive/Dataset/rooms_dataset/living_room/photo-1554995207-c18c203602cb (1).jpg'), ('living_room', '/content/drive/MyDrive/Dataset/rooms_dataset/living_room/pexels-photo-1571470.jpeg'), ('living_room', '/content/drive/M

In [5]:
room_df = pd.DataFrame(data = rooms, columns=['room type','image'])
room_df.head()

,room type,image
0,living_room,/content/drive/MyDrive/Dataset/rooms_dataset/l...
1,living_room,/content/drive/MyDrive/Dataset/rooms_dataset/l...
2,living_room,/content/drive/MyDrive/Dataset/rooms_dataset/l...
3,living_room,/content/drive/MyDrive/Dataset/rooms_dataset/l...
4,living_room,/content/drive/MyDrive/Dataset/rooms_dataset/l...


In [6]:
print("Total number of rooms in the dataset: ", len(room_df))

room_count = room_df['room type'].value_counts()
print(room_count)

Total number of rooms in the dataset:  118
room type
bed_room       46
living_room    38
dining_room    34
Name: count, dtype: int64


In [10]:
import cv2 as cv
path = "/content/drive/MyDrive/Dataset/rooms_dataset"

img_size = 224
images = []
labels = []

for i in room_type:
    data_path = path + "/" + str(i)
    filenames = [i for i in os.listdir(data_path)]

   # print(filenames) ## images name only
    for f in filenames:
        img = cv.imread(data_path+"/"+f)

        img = cv.resize(img, (img_size,img_size))
        images.append(img)
        labels.append(i)

labels

['living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining

In [11]:
images = np.array(images)
images.shape

(118, 224, 224, 3)

In [12]:
images = images.astype('float32')/255.0

In [13]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

y = room_df['room type'].values

y_labelEncoder = LabelEncoder()
y = y_labelEncoder.fit_transform(y)

y = y.reshape(-1,1)
onehotencoder = OneHotEncoder(sparse_output=False)
y = onehotencoder.fit_transform(y)
y.shape

(118, 3)

In [14]:
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split

images , y = shuffle(images, y , random_state=1)
x_train, x_test, y_train, y_test = train_test_split(images, y, test_size=0.05, random_state=415)

print(x_train.shape, x_test.shape)
print(y_train.shape, y_test.shape)


(112, 224, 224, 3) (6, 224, 224, 3)
(112, 3) (6, 3)


## building model

In [24]:
import tensorflow
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPooling2D, Input, BatchNormalization, Activation, ZeroPadding2D, AveragePooling2D, Add
from tensorflow.keras.preprocessing import image
from keras.initializers import glorot_uniform

## Identity Block


In [18]:
def identity_block(X, f, filters):

    # Retrieve Filters
    F1, F2, F3 = filters  # F1=64,F2=64,256

    X_shortcut = X

    # First  layer
    X = Conv2D(filters = F1, kernel_size = (1, 1), strides = (1,1), padding = 'valid')(X)
    X = BatchNormalization(axis = 3)(X)
    X = Activation('relu')(X)


    # Second  layer
    X = Conv2D(filters = F2, kernel_size = (f, f), strides = (1,1), padding = 'same')(X)
    X = BatchNormalization(axis = 3)(X)
    X = Activation('relu')(X)

    # Third  layer
    X = Conv2D(filters = F3, kernel_size = (1, 1), strides = (1,1), padding = 'valid')(X)
    X = BatchNormalization(axis = 3)(X)

    # Final step: Add shortcut value to F(X), and pass it through a RELU activation
    X = Add()([X, X_shortcut])
    X = Activation('relu')(X)


    return X

## Convolutional Block

In [19]:
def convolutional_block(X, f, filters, s = 2):


    # Retrieve Filters
    F1, F2, F3 = filters

    # Save the input value
    X_shortcut = X


    # First layer
    X = Conv2D(F1, (1, 1), strides = (s,s))(X) # 1,1 is filter size
    X = BatchNormalization(axis = 3)(X)  # normalization on channels
    X = Activation('relu')(X)


    # Second layer  (f,f)=3*3 filter by default
    X = Conv2D(filters = F2, kernel_size = (f, f), strides = (1,1), padding = 'same')(X)
    X = BatchNormalization(axis = 3)(X)
    X = Activation('relu')(X)


    # Third layer
    X = Conv2D(filters = F3, kernel_size = (1, 1), strides = (1,1), padding = 'valid')(X)
    X = BatchNormalization(axis = 3)(X)


    ##### SHORTCUT PATH ####
    X_shortcut = Conv2D(filters = F3, kernel_size = (1, 1), strides = (s,s), padding = 'valid')(X_shortcut)
    X_shortcut = BatchNormalization(axis = 3)(X_shortcut)

    # Final step: Add shortcut value here, and pass it through a RELU activation
    X = Add()([X, X_shortcut])
    X = Activation('relu')(X)


    return X


In [25]:
def ResNet50(input_shape=(224, 224, 3), classes=3):
    """
    Implementation of the ResNet50 architecture:
    CONV2D -> BATCHNORM -> RELU -> MAXPOOL -> CONVBLOCK -> IDBLOCK*2 -> CONVBLOCK -> IDBLOCK*3
    -> CONVBLOCK -> IDBLOCK*5 -> CONVBLOCK -> IDBLOCK*2 -> AVGPOOL -> TOPLAYER

    """

    # Define the input with shape input_shape
    X_input = Input(input_shape)

    # Zero-Padding
    X = ZeroPadding2D((3, 3))(X_input) #3,3 padding
    print(X.shape,"shape of image before stage 1 after adding zero padding conv layer")
    # Stage 1

    X = Conv2D(64, (7, 7), strides=(2, 2))(X)
    print(X.shape,"shape of image at stage 1 after conv layer")
    X = BatchNormalization(axis=3)(X)
    X = Activation('relu')(X)
    X = MaxPooling2D((3, 3), strides=(2, 2))(X)

    # Stage 2
    X = convolutional_block(X, f=3, filters=[64, 64, 256], s=1)

    # below 3 lines are the conv layers from convolutional_block function defined above
    #X = Conv2D(F1, (1, 1), strides = (s,s))(X)
    #X = Conv2D(F2, kernel_size = (f, f), strides = (1,1), padding = 'same')(X)
    #X = Conv2D(F3, (1, 1), strides = (s,s), name = conv_name_base + '2a')(X)

    X = identity_block(X, 3, [64, 64, 256])
    #X = Conv2D(filters = F1, kernel_size = (1, 1), strides = (1,1), padding = 'valid')(X)
    #X = Conv2D(filters = F2, kernel_size = (f, f), strides = (1,1), padding = 'same')(X)
    #X = Conv2D(filters = F3, kernel_size = (1, 1), strides = (1,1), padding = 'valid')(X)

    X = identity_block(X, 3, [64, 64, 256])
    #X = Conv2D(filters = F1, kernel_size = (1, 1), strides = (1,1), padding = 'valid')(X)
    #X = Conv2D(filters = F2, kernel_size = (f, f), strides = (1,1), padding = 'same')(X)
    #X = Conv2D(filters = F3, kernel_size = (1, 1), strides = (1,1), padding = 'valid')(X)


    # Stage 3
    X = convolutional_block(X, f = 3, filters = [128, 128, 512], s = 2)
    X = identity_block(X, 3, [128, 128, 512])
    X = identity_block(X, 3, [128, 128, 512])
    X = identity_block(X, 3, [128, 128, 512])

    # Stage 4
    X = convolutional_block(X, f = 3, filters = [256, 256, 1024], s = 2)
    X = identity_block(X, 3, [256, 256, 1024])
    X = identity_block(X, 3, [256, 256, 1024])
    X = identity_block(X, 3, [256, 256, 1024])
    X = identity_block(X, 3, [256, 256, 1024])
    X = identity_block(X, 3, [256, 256, 1024])

    # Stage 5
    X = convolutional_block(X, f = 3, filters = [512, 512, 2048], s = 2)
    X = identity_block(X, 3, [512, 512, 2048])
    X = identity_block(X, 3, [512, 512, 2048])

    # AVGPOOL
    X = AveragePooling2D((2,2), name="avg_pool")(X)

    ### END CODE HERE ###

    # output layer
    X = Flatten()(X)
    X = Dense(classes, activation='softmax', name='fc' + str(classes), kernel_initializer = glorot_uniform(seed=0))(X)


    # Create model
    model = Model(inputs = X_input, outputs = X, name='ResNet50')

    return model

In [26]:
ResNet50(input_shape=(224, 224, 3), classes=3)

(None, 230, 230, 3) shape of image before stage 1 after adding zero padding conv layer
(None, 112, 112, 64) shape of image at stage 1 after conv layer


<Functional name=ResNet50, built=True>

In [27]:
model = ResNet50(input_shape = (224, 224, 3), classes = 3)

(None, 230, 230, 3) shape of image before stage 1 after adding zero padding conv layer
(None, 112, 112, 64) shape of image at stage 1 after conv layer


In [28]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [29]:
model.summary()

Model: "ResNet50"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d_2    │ (None, 230, 230,  │          0 │ input_layer_2[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_58 (Conv2D)  │ (None, 112, 112,  │      9,472 │ zero_padding2d_2… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 112, 112,  │        256 │ conv2d_58[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_52       │ (None, 112, 112,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 55, 55,    │          0 │ activation_52[0]… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_59 (Conv2D)  │ (None, 55, 55,    │      4,160 │ max_pooling2d_2[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 55, 55,    │        256 │ conv2d_59[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_53       │ (None, 55, 55,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_60 (Conv2D)  │ (None, 55, 55,    │     36,928 │ activation_53[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 55, 55,    │        256 │ conv2d_60[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_54       │ (None, 55, 55,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_61 (Conv2D)  │ (None, 55, 55,    │     16,640 │ activation_54[0]… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_62 (Conv2D)  │ (None, 55, 55,    │     16,640 │ max_pooling2d_2[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 55, 55,    │      1,024 │ conv2d_61[0][0]   │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 55, 55,    │      1,024 │ conv2d_62[0][0]   │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_16 (Add)        │ (None, 55, 55,    │          0 │ batch_normalizat

 Total params: 23,643,011 (90.19 MB)

 Trainable params: 23,589,891 (89.99 MB)

 Non-trainable params: 53,120 (207.50 KB)

In [31]:
model.fit(x_train, y_train ,epochs = 10, batch_size = 32)

Epoch 1/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 119s 13s/step - accuracy: 0.2500 - loss: 19.2483
Epoch 2/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 338ms/step - accuracy: 0.4107 - loss: 5.0787
Epoch 3/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 293ms/step - accuracy: 0.3304 - loss: 3.3458
Epoch 4/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 324ms/step - accuracy: 0.4821 - loss: 1.4562
Epoch 5/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 273ms/step - accuracy: 0.4821 - loss: 1.3173
Epoch 6/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 278ms/step - accuracy: 0.5268 - loss: 1.4447
Epoch 7/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 319ms/step - accuracy: 0.7054 - loss: 0.9534
Epoch 8/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 278ms/step - accuracy: 0.6786 - loss: 1.1992
Epoch 9/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 252ms/step - accuracy: 0.7411 - loss: 0.7416
Epoch 10/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 252ms/step - accuracy: 0.7679 - loss: 1.0093


In [33]:
preds = model.evaluate(x_test, y_test)
print ("Loss = " + str(preds[0]))
print ("Test Accuracy = " + str(preds[1]))

1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step - accuracy: 0.3333 - loss: 1.2969
Loss = 1.296878457069397
Test Accuracy = 0.3333333432674408


In [35]:
pred = model.predict(x_test)
pred = pred.argmax(axis =1)

y_test_labels = y_test.argmax(axis=1) # Convert one-hot encoded y_test to integer labels

from sklearn.metrics import accuracy_score
print(accuracy_score(y_test_labels, pred))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
0.3333333333333333
